In [1]:
import numpy as np
import os
from scipy.io import savemat
import pandas as pd
from pathlib import Path

In [ ]:
def extract_recordings(recording_data,ground_truth_data,dir_to_save_output,list_of_channels_to_save):

    try:
        os.makedirs(dir_to_save_output)
    except OSError as e:
        print(f"Directory {dir_to_save_output} already exists or could not be created: {e}")

    try:
        os.makedirs(os.path.join(dir_to_save_output,"recordings_by_channel"))
    except OSError as e:
        print(f"Directory {dir_to_save_output} already exists or could not be created: {e}")

    try:
        os.makedirs(os.path.join(dir_to_save_output,"timestamps"))
    except OSError as e:
        print(f"Directory {dir_to_save_output} already exists or could not be created: {e}")

    try:
        os.makedirs(os.path.join(dir_to_save_output,"ground_truth"))
    except OSError as e:
        print(f"Directory {dir_to_save_output} already exists or could not be created: {e}")

    
    for i in list_of_channels_to_save:
        channel_data = recording_data[i-1,:] * 2.34 # Convert to microvolts as per github instructions
        channel_data = channel_data.reshape(-1, 1) # Reshape to 2D array with 1 row and N columns
        mat_dict = {f"c_{i}": channel_data}
        savemat(os.path.join(dir_to_save_output,"recordings_by_channel",f"c{i-1}.mat"), mat_dict)
        print(f"Saved channel {i} data to {os.path.join(dir_to_save_output,'recordings_by_channel',f'channel_{i}.mat')}")

    timestamps = timestamps = np.arange(recording_data.shape[1]) / 30000
    savemat(os.path.join(dir_to_save_output,"timestamps","timestamps.mat"), {"timestamps": timestamps})
    print(f"Saved timestamps to {os.path.join(dir_to_save_output,'timestamps','timestamps.mat')}")

   

    ground_truth_dict = {"spike_trains": ground_truth_data};
    savemat(os.path.join(dir_to_save_output,"ground_truth","ground_truth.mat"), ground_truth_dict)
    print(f"Saved ground truth to {os.path.join(dir_to_save_output,'ground_truth','ground_truth.mat')}")

In [3]:
def list_files_pathlib(directory_path, extension):
    """
    Recursively lists all files with a specific extension using pathlib.
    """
    # Create a Path object for the starting directory
    p = Path(directory_path)
    # Use rglob (recursive glob) to find matching files
    # The pattern should be f'*.{extension}'
    files = list(p.rglob(f'*.{extension}'))
    return files

# Read The data Summary file

In [4]:
data_summary = pd.read_csv("F:\Data Summary.csv");
data_summary

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\ldd77\AppData\Local\Temp\ipykernel_11952\2074656916.py:1: SyntaxWarning: invalid escape sequence '\D'
  data_summary = pd.read_csv("F:\Data Summary.csv");


,Cell,cell order(day),Date,Session,Probe Insert x,Probe Insert y,Probe Insert z,Probe Insert L,Ratio to Paxinos,Cell x,...,chan_predicted,JTA Peak-Peak Amplitude,chan_highest,Patch Type,# Patch Spikes,Recording Length (mins),Spike T,# channels pk >20µV,Cell Type,Patch Samp. Freq. (Hz)
0,c1,1,8/31/2017,1,-2461.2,2527.4,2847.6,3000.0,NaN,-2022.9,...,204,4.7,221,"Juxta, IC",3019,4.50,1.25,0,PC,50023.918758
1,c2,2,8/31/2017,1,-2461.2,2527.4,2847.6,3000.0,NaN,-2110.0,...,219,16.4,221,"Juxta, VC",835,6.80,62.2,0,PC,50023.912582
2,c3,3,8/31/2017,1,-2461.2,2527.4,2847.6,3000.0,NaN,-1991.7,...,188,4.8,302,"Juxta, IC",399,6.75,0.81,0,PC,50023.916699
3,c4,1,9/6/2017,2,-2400.1,2487.5,2775.0,3000.0,NaN,-1961.2,...,204,21.1,203,"WC, IC",1335,12.60,2.8,1,PC,50023.911259
4,c5,2,9/6/2017,3,-2099.7,2253.4,2708.9,3000.0,NaN,-1857.8,...,251,4.7,202,"WC, IC",3589,27.00,10.4,0,PC,50023.907436
5,c6,3,9/6/2017,3,-2099.7,2253.4,2708.9,3000.0,NaN,-1540.5,...,172,18.8,157,"Juxta, VC",9395,27.00,85,0,PC,50023.898173
6,c7,1,9/14/2017,4,-2732.7,1275.0,2756.0,3000.0,NaN,-2247.6,...,188,11.7,192,"Juxta, IC",11998,27.00,10.5,0,IN,50023.913612
7,c8,2,9/14/2017,4,-2732.7,1275.0,2756.0,3000.0,NaN,-2264.0,...,188,9.4,170,"Juxta, IC",4890,27.00,3.41,0,IN,50023.909495
8,c10,4,9/14/2017,4,-2732.7,1275.0,2756.0,3000.0,NaN,-2389.0,...,219,3.2,193,"Juxta, VC",1092,6.75,65.1,0,PC,50023.900231
9,c12,2,11/28/2017,5,-3000.0,2340.0,1630.0,2520.3,NaN,-2493.6,...,138,11.7,133,"Juxta, VC",7399,27.00,75,0,IN,50023.898173


# Get a list of all available cell data

In [53]:
list_of_cell_data_available = list_files_pathlib

# check which c1 files we have downloaded

In [54]:
list_of_all_files = list_files_pathlib(".","_npx_raw.bin")
list_of_all_files

[]

# Get the neuropixel data

In [5]:
npx_path = "F:\c1\c1\c1_npx_raw.bin"
npx_channels = 384

npx_recording = np.memmap( npx_path, mode = 'r', dtype=np.int16, order = 'C')

npx_samples = int(len(npx_recording)/npx_channels)

npx_recording = npx_recording.reshape((npx_channels, npx_samples), order = 'F')
print(npx_recording.shape)

(384, 8100337)


<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\ldd77\AppData\Local\Temp\ipykernel_11952\2981904836.py:1: SyntaxWarning: invalid escape sequence '\c'
  npx_path = "F:\c1\c1\c1_npx_raw.bin"


# Get the recording patch data

In [6]:
patch_path = "F:\c1\c1\c1_patch_ch1.bin"
patch_recording = np.fromfile(patch_path, dtype='float64')

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\ldd77\AppData\Local\Temp\ipykernel_11952\835974785.py:1: SyntaxWarning: invalid escape sequence '\c'
  patch_path = "F:\c1\c1\c1_patch_ch1.bin"


# Get the conversion factor

In [7]:
m = len(patch_recording)/float( len(npx_recording[0]))

#neuropixel_event = (patch_recording / m).astype(int)
#patch_event = (npx_recording * m).astype(int)

# The GT Idx in Patch

In [16]:
file_path = "F:\c1\c1\c1_wc_spike_samples.npy"

# Load the data from the .npy file
ground_truth_idxs_in_patch_format = np.load(file_path)

# You can now use the 'data' array like any other NumPy array
print("Data loaded successfully:")
print(ground_truth_idxs_in_patch_format)
print("Shape of the data:", ground_truth_idxs_in_patch_format.shape)
print("Data type:", ground_truth_idxs_in_patch_format.dtype)
print("Converting to neuropixel format...")
ground_truth_idxs_as_timestamps = ground_truth_idxs_in_patch_format / 50023.9187579479
ground_truth_idxs_in_neuropixel_format = np.round(ground_truth_idxs_in_patch_format / m).astype(np.int64)
print("Conversion complete.")

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\ldd77\AppData\Local\Temp\ipykernel_11952\1187912420.py:1: SyntaxWarning: invalid escape sequence '\c'
  file_path = "F:\c1\c1\c1_wc_spike_samples.npy"


Data loaded successfully:
[   10431    11427    12020 ... 13468180 13471832 13491165]
Shape of the data: (3019,)
Data type: int64
Converting to neuropixel format...
Conversion complete.


C:\Users\ldd77\AppData\Local\Temp\ipykernel_11952\1187912420.py:4: UserWarning: Reading `.npy` or `.npz` file required additional header parsing as it was created on Python 2. Save the file again to speed up loading and avoid this warning.
  ground_truth_idxs_in_patch_format = np.load(file_path)


In [75]:
ground_truth_idxs_in_patch_format

array([   10431,    11427,    12020, ..., 13468180, 13471832, 13491165],
      dtype=int64)

In [74]:
ground_truth_idxs_in_neuropixel_format

array([   6255,    6852,    7208, ..., 8077044, 8079234, 8090828])

In [18]:
#extract_recordings(npx_recording,ground_truth_idxs_as_timestamps,"F:\cell_1",[124, 125, 126, 220, 221, 222, 316, 317, 318])
extract_recordings(npx_recording,ground_truth_idxs_in_neuropixel_format,"F:\cell_1",range(0,383,1))

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
C:\Users\ldd77\AppData\Local\Temp\ipykernel_11952\3751051885.py:2: SyntaxWarning: invalid escape sequence '\c'
  extract_recordings(npx_recording,ground_truth_idxs_in_neuropixel_format,"F:\cell_1",range(0,383,1))


Directory F:\cell_1 already exists or could not be created: [WinError 183] Cannot create a file when that file already exists: 'F:\\cell_1'
Directory F:\cell_1 already exists or could not be created: [WinError 183] Cannot create a file when that file already exists: 'F:\\cell_1\\recordings_by_channel'
Directory F:\cell_1 already exists or could not be created: [WinError 183] Cannot create a file when that file already exists: 'F:\\cell_1\\timestamps'
Directory F:\cell_1 already exists or could not be created: [WinError 183] Cannot create a file when that file already exists: 'F:\\cell_1\\ground_truth'
Saved channel 0 data to F:\cell_1\recordings_by_channel\channel_0.mat
Saved channel 1 data to F:\cell_1\recordings_by_channel\channel_1.mat
Saved channel 2 data to F:\cell_1\recordings_by_channel\channel_2.mat
Saved channel 3 data to F:\cell_1\recordings_by_channel\channel_3.mat
Saved channel 4 data to F:\cell_1\recordings_by_channel\channel_4.mat
Saved channel 5 data to F:\cell_1\recordi